
AI Agent

Last Updated: April 29th, 2025

Daily Challenge : AI Agent for Emergency Medical Dispatch


👩‍🏫 👩🏿‍🏫 What You'll learn

You'll deepen your understanding of how AI agents perceive, reason, and act by analyzing their architecture and designing your own. You'll also critically compare agent types and decide which is best for specific tasks.


🛠️ What you will create

You'll design a smart emergency dispatch agent that assists in triaging 911 calls, gathering critical patient data, and dispatching appropriate medical services. You'll define its architecture, tools, state management, and evaluate the trade-offs between different agent types.


🛠 What will you use

    Agent concepts: perception, reasoning, action loops
    Architecture types: reactive, deliberative, hybrid
    Tools & integrations: symptom‐checker APIs, dispatch scheduling systems, GIS/location services
    State management: memory of caller info, symptom history, decision logs
    Evaluation criteria: urgency determination, response speed, reliability, safety trade-offs


Step-by-Step Instructions

1. Understand the Scenario

    Read the challenge description.
    Note the goal: build an AI assistant that triages 911 calls and dispatches medical help.

2. Define the Agent's Environment

    List all inputs your agent will “perceive”:
        Voice or text transcript of caller's symptoms
        Caller location (GPS/address)
        Caller identity and medical history (if available)
    Write down each input as bullet points in your notes.

3. Select and Describe Tools

    Identify at least three external systems or APIs your agent needs:
        Symptom Checker API (e.g., Infermedica)
        Ambulance Scheduling System (e.g., internal dispatch API)
        Medical Triage Model (e.g., fine-tuned LLM for severity scoring)
    For each tool, note: what data it consumes, and what it returns.

4. Outline State Management

    Decide what the agent must remember across the call:
        Caller identity and contact info
        Reported symptoms and their severity
        Decisions/actions taken so far (e.g., advised self-care, dispatched ambulance)
    Sketch a simple state table or JSON schema listing these fields.

5. Design the Decision-Making Process

    Break down urgency determination into steps:
        Parse symptoms → extract severity keywords
        Query triage model → get “urgency score”
        Compare score against thresholds → “High”, “Medium”, “Low”
    Define what the agent does at each level:
        High → dispatch ambulance immediately
        Medium → advise nearest urgent care
        Low → provide self-care instructions

6. Classify Your Agent

    Choose one architecture type (Reactive, Deliberative, or Hybrid).
    Justify:
        How does it use memory?
        Does it plan ahead or act on immediate inputs?
        Example justification in 2–3 sentences.

7. Compare to a Second Agent Type

    Pick a different type (e.g., if you chose Hybrid, compare to Reactive).
    Describe how your design would change:
        Memory handling
        Planning steps or lack thereof
        Tool invocation strategy
    List trade-offs in speed, reliability, and intelligence (one bullet each).

8. Reflect on Critical Questions

    What fails if your agent does not maintain state?
    Why are external tools (APIs, models) essential in an EMR dispatch scenario?
    Write 2–3 sentences for each reflection question.


Deliverables

1. Design Document (Markdown or Notebook)

    Environment definition
    Tool list & interfaces
    State schema
    Decision-making flowchart or pseudocode

2. Agent Classification & Comparison

    Chosen architecture with justification
    Comparison to second type with trade-off analysis

3. Reflection Answers

    Impact of no state management
    Role of tools in high-stakes dispatch

4. (Optional) GitHub Submission

    Push your final .md or notebook to a public repo and share the link.


Il y a dans ce projet des fichiers séparés afin de tester les appel via client / streamablehttp_client. client/agent/server test la logique avec protocole MCP . Agent_client / agent_tools et server v2 test la même logique avec un llm llama 3.1 

Ce notebook répond point par point à l'énoncé :  
- Définition de l'environnement
- Liste des outils/API utilisés
- Schéma de gestion d'état
- Logique de décision
- Classification d'agent
- Réflexion sur les limites “AI”
- Test automatisé

- Transcription texte (ou voix) des symptômes du patient
- Localisation du patient (adresse, GPS)
- Identité de l'appelant (si connue) et antécédents médicaux (si disponibles)

**Inputs principaux :**
- symptoms: str
- location: str
- caller_id: str (optionnel)

In [ ]:
# Outils/API simulés pour l'agent

def symptom_checker(symptoms: str) -> dict:
    """
    Simule un appel à une API externe qui retourne un score de gravité.
    """
    # Ici, on fait simple : score élevé si on parle de "douleur thoracique"
    if "douleur thoracique" in symptoms:
        return {"urgency_score": 8, "details": "douleur thoracique 8/10"}
    elif "fièvre" in symptoms:
        return {"urgency_score": 4, "details": "fièvre modérée"}
    else:
        return {"urgency_score": 2, "details": "symptôme bénin"}

def dispatch_ambulance(location: str, caller_id: str = None) -> str:
    return f"Ambulance envoyée à {location}"

def urgent_care(location: str) -> str:
    return f"Centre d'urgent care à proximité de {location}"

def self_care(symptoms: str) -> str:
    return f"Conseils d'auto-soin pour : {symptoms}"

In [ ]:
# Gestion simple de l'état sous forme de dictionnaire (persistant en RAM ici)
state_store = {}

def save_state(tool_use_id: str, state: dict):
    state_store[tool_use_id] = state

def load_state(tool_use_id: str):
    return state_store.get(tool_use_id, None)

In [ ]:
HIGH_THRESHOLD = 7
MEDIUM_THRESHOLD = 4

def triage_call(symptoms: str, location: str, caller_id: str = None, tool_use_id: str = None) -> str:
    # 1. Récupération du score de gravité
    result = symptom_checker(symptoms)
    score = result["urgency_score"]

    # 2. Décision en fonction du score
    if score >= HIGH_THRESHOLD:
        action = dispatch_ambulance(location, caller_id)
    elif score >= MEDIUM_THRESHOLD:
        action = urgent_care(location)
    else:
        action = self_care(symptoms)

    # 3. Mise à jour de l'état
    if tool_use_id:
        save_state(tool_use_id, {
            "symptoms": symptoms,
            "score": score,
            "action": action
        })

    return action

In [5]:
# Test 1 : cas grave
symptoms = "douleur thoracique et essoufflement"
location = "123 Rue de la Paix, Paris"
caller_id = "appelant_42"
tool_use_id = "triage-0001"

result = triage_call(symptoms, location, caller_id, tool_use_id)
print("Test 1 (cas grave) :", result)
print("État enregistré :", state_store[tool_use_id])

# Test 2 : cas modéré
symptoms2 = "fièvre et toux"
tool_use_id2 = "triage-0002"
result2 = triage_call(symptoms2, location, caller_id, tool_use_id2)
print("Test 2 (modéré) :", result2)
print("État enregistré :", state_store[tool_use_id2])

# Test 3 : cas bénin
symptoms3 = "mal de tête"
tool_use_id3 = "triage-0003"
result3 = triage_call(symptoms3, location, caller_id, tool_use_id3)
print("Test 3 (bénin) :", result3)
print("État enregistré :", state_store[tool_use_id3])

Test 1 (cas grave) : Ambulance envoyée à 123 Rue de la Paix, Paris
État enregistré : {'symptoms': 'douleur thoracique et essoufflement', 'score': 8, 'action': 'Ambulance envoyée à 123 Rue de la Paix, Paris'}
Test 2 (modéré) : Centre d’urgent care à proximité de 123 Rue de la Paix, Paris
État enregistré : {'symptoms': 'fièvre et toux', 'score': 4, 'action': 'Centre d’urgent care à proximité de 123 Rue de la Paix, Paris'}
Test 3 (bénin) : Conseils d’auto-soin pour : mal de tête
État enregistré : {'symptoms': 'mal de tête', 'score': 2, 'action': 'Conseils d’auto-soin pour : mal de tête'}



Architecture choisie : Agent hybride
- Il utilise la mémoire (état sauvegardé)
- Il suit une boucle perception → raisonnement → action
- Il prend une décision basée sur le contexte ET sur les entrées immédiates

L'agent combine des éléments réactifs (if/else selon le score des symptômes) et délibératifs (mémorisation de chaque décision, possibilité de suivre un historique). Il planifie peu, mais son architecture lui permettrait de s'adapter si on branchait un vrai moteur IA.

Comparaison avec un agent réactif pur

- Réactif pur : ne garde aucune mémoire, chaque appel est indépendant, agit directement selon l'entrée.
- Hybride (ici) : garde un état, pourrait adapter son comportement si le contexte évolue.
    - Vitesse : agent réactif = plus rapide, moins coûteux
    - Intelligence/adaptation : hybride > réactif
    - Fiabilité : hybride peut mieux gérer l'historique et éviter les répétitions

### Que se passe-t-il si l'agent ne garde pas d'état ?

Sans gestion d'état, l'agent ne pourrait pas suivre l'évolution de l'appel ni adapter ses réponses selon les interactions passées. Il risquerait de répéter les mêmes conseils, de ne pas détecter une aggravation, ou d'oublier une action déjà prise.

### Pourquoi les outils externes sont-ils essentiels ?

Les outils (APIs médicales, dispatch, scoring) sont indispensables car ils permettent à l'agent de s'appuyer sur des modèles experts, des bases médicales à jour, ou des systèmes opérationnels (dispatch). Sans ces outils, l'agent ne pourrait ni bien estimer la gravité, ni agir de manière coordonnée dans un vrai contexte d'urgence.

### Limites de la logique “AI” ici

Dans cette version, la “logique AI” est limitée à une simple cascade de règles if/else. Ce n'est pas une IA forte, mais une architecture d'agent prête à intégrer des modules plus intelligents (ex : API IA médicale). Ce projet illustre l'organisation d'un agent, non la performance d'un vrai modèle AI.

Ce notebook illustre la structure et le fonctionnement d’un agent intelligent pour le dispatch médical d’urgence.  
La logique reste simple (règles conditionnelles), mais l’architecture permettrait facilement d’intégrer une vraie IA médicale par la suite.